# Data Science Experiment 5 – Feature Selection

Using `urban_climate.csv` to select features for predicting `urban_heat_island_intensity`. The target is continuous, so this is a regression task.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUT = Path('outputs')
OUTPUT.mkdir(exist_ok=True)


## Steps 1–3: Load data, define X/y and clean

In [ ]:
df = pd.read_csv('urban_climate.csv')
print('Shape:', df.shape)
display(df.head())
print('Missing values:\n', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
TARGET = 'urban_heat_island_intensity'
feature_cols = ['latitude','longitude','year','month','temperature_celsius','humidity_percent','precipitation_mm','wind_speed_ms','country_population_reference','country_area_km2']
X = df[feature_cols].replace([np.inf,-np.inf], np.nan).copy()
y = df[TARGET].copy()
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())
valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)


## Train-test split

In [ ]:
train_idx, test_idx = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()


## Step 4: Correlation analysis

In [ ]:
train_corr = X_train.copy()
train_corr[TARGET] = y_train.values
corr_matrix = train_corr.corr(numeric_only=True)
corr_rank = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
display(corr_rank)
plt.figure(figsize=(11,8)); plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1); plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90); plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('Correlation Heatmap'); plt.tight_layout(); plt.savefig(OUTPUT/'01_correlation_heatmap.png', dpi=160); plt.show()


## Step 5: Filter method – Mutual Information

In [ ]:
mi_values = mutual_info_regression(X_train, y_train, random_state=42)
mi_rank = pd.Series(mi_values, index=feature_cols).sort_values(ascending=False)
display(mi_rank)
mi_rank.sort_values().plot(kind='barh', figsize=(9,6), title='Mutual Information Feature Ranking')
plt.xlabel('Mutual information'); plt.tight_layout(); plt.savefig(OUTPUT/'02_mutual_information.png', dpi=160); plt.show()


## Step 6: Wrapper method – Recursive Feature Elimination

In [ ]:
rfe = RFE(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), n_features_to_select=5, step=1)
rfe.fit(X_train, y_train)
rfe_features = X_train.columns[rfe.support_].tolist()
rfe_rank = pd.DataFrame({'Feature':feature_cols, 'Selected':rfe.support_, 'RFE_Rank':rfe.ranking_}).sort_values(['Selected','RFE_Rank'], ascending=[False,True])
display(rfe_rank)
rfe_rank.to_csv(OUTPUT/'rfe_feature_ranking.csv', index=False)


## Step 7: Embedded method – Random Forest feature importance

In [ ]:
embedded_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
embedded_model.fit(X_train, y_train)
importance = pd.Series(embedded_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(importance)
importance.sort_values().plot(kind='barh', figsize=(9,6), title='Random Forest Feature Importance')
plt.xlabel('Importance'); plt.tight_layout(); plt.savefig(OUTPUT/'03_random_forest_importance.png', dpi=160); plt.show()


## Step 8: Compare selected features

In [ ]:
comparison = pd.DataFrame({'Feature':feature_cols, 'Correlation_Top5':[f in corr_rank.head(5).index for f in feature_cols], 'Mutual_Info_Top5':[f in mi_rank.head(5).index for f in feature_cols], 'RFE_Selected':[f in rfe_features for f in feature_cols], 'Random_Forest_Top5':[f in importance.head(5).index for f in feature_cols]})
comparison['Selection_Count'] = comparison.iloc[:,1:5].sum(axis=1)
comparison = comparison.sort_values(['Selection_Count','Feature'], ascending=[False,True])
display(comparison)
comparison.to_csv(OUTPUT/'feature_selection_comparison.csv', index=False)
selected_features = rfe_features
print('RFE-selected features:', selected_features)


## Steps 9–11: Train and compare models

In [ ]:
def evaluate_model(model_features, model_name):
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    start = time.perf_counter(); model.fit(X_train[model_features], y_train); train_time = time.perf_counter()-start
    start = time.perf_counter(); pred = model.predict(X_test[model_features]); predict_time = time.perf_counter()-start
    return {'Model':model_name, 'Number_of_Features':len(model_features), 'R2_Score':r2_score(y_test,pred), 'MAE':mean_absolute_error(y_test,pred), 'RMSE':mean_squared_error(y_test,pred)**0.5, 'Training_Time_Seconds':train_time, 'Prediction_Time_Seconds':predict_time}
results = pd.DataFrame([evaluate_model(feature_cols,'All Features'), evaluate_model(selected_features,'RFE Selected Features')])
display(results)
results.to_csv(OUTPUT/'model_performance_comparison.csv', index=False)


In [ ]:
# Model comparison graph: predictive performance and computational time
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(results['Model'], results['R2_Score'])
axes[0].set_title('Model Comparison - R² Score')
axes[0].set_ylabel('R² (higher is better)')
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(results['Model'], results['Training_Time_Seconds'])
axes[1].set_title('Model Comparison - Training Time')
axes[1].set_ylabel('Training time (seconds)')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT/'04_model_comparison_graph.png', dpi=160)
plt.show()


## Final interpretation

In [ ]:
print('Target:', TARGET)
print('Original features:', len(feature_cols))
print('Selected features:', selected_features)
print('Compare R2 (higher is better), MAE/RMSE (lower is better), and execution times.')
